**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Sigma-Delta & Quantization Theory

How a **1-bit** converter delivers 16+ bits of audio: oversampling spreads quantization noise thin, and the ΣΔ loop *shapes* it out of the band you care about. The bridge between [Real-Time DSP's](./Real_Time_DSP.ipynb) fixed-point world and actual converter hardware — with the 6 dB/bit and noise-shaping laws measured, not recited.

## 1. Pre-requisites

[Real-Time DSP](./Real_Time_DSP.ipynb) S1 (Q-format), [FoSP2](./Foundations_of_Signal_Processing_2.ipynb) S2 (oversampling/decimation), [Statistical SP](./Statistical_Signal_Processing.ipynb) S2 (PSD).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as sig
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 2 — *Quantization Noise & the Oversampling Dividend* (~40 min)
**Goal:** the white-noise model of quantization, its 6 dB/bit law, and +3 dB per octave of oversampling.
**Builds on:** [Real-Time DSP](./Real_Time_DSP.ipynb) S1. &nbsp; **Feeds into:** Session 2 (noise shaping).

---

## 2. Noise You Can Dilute

💡 **Intuition.** Quantization error behaves like white noise of total power $\Delta^2/12$ — and that total is **fixed by the step size**, not by the sampling rate. Sample $M×$ faster and the same noise power spreads over $M×$ the bandwidth: the slice sitting on your signal band shrinks by $M$ — **+3 dB of SNR per octave of oversampling** (half a bit per octave), redeemed by a [decimating low-pass filter](./Foundations_of_Signal_Processing_2.ipynb). Useful, but slow: 16 bits from 1 bit would need $2^{30}$× oversampling. Session 2 does radically better.

In [2]:
# the two laws, measured
def quantize(x, bits):
    L = 2**(bits-1)
    return np.clip(np.round(x*L), -L, L-1)/L

fs_base = 48_000
t = np.arange(2**16)/fs_base
x = 0.9*np.sin(2*np.pi*997*t)                        # 997 Hz: coprime-ish with fs, avoids spectral pileup

print("6 dB/bit (measured in the full band):")
for b in [4, 8, 12]:
    e = quantize(x, b) - x
    print(f"  {b:2d} bits: SNR {10*np.log10(np.var(x)/np.var(e)):5.1f} dB   (theory ≈ {6.02*b+1.76:.1f})")

print("\n+3 dB per octave of oversampling (8-bit quantizer, 24 kHz signal band):")
for M in [1, 4, 16]:
    fs_os = fs_base*M
    t_os = np.arange(2**16)/fs_os
    x_os = 0.9*np.sin(2*np.pi*997*t_os)
    q = quantize(x_os, 8)
    # measure noise power INSIDE the audio band only
    f, P = sig.welch(q - x_os, fs=fs_os, nperseg=4096)
    inband = P[f < fs_base/2].sum() * (f[1]-f[0])
    print(f"  {M:2d}x oversampled: in-band quantization noise {10*np.log10(inband):6.1f} dB   "
          f"({0 if M==1 else -10*np.log10(M):+.1f} dB expected shift)")

6 dB/bit (measured in the full band):
   4 bits: SNR  25.5 dB   (theory ≈ 25.8)
   8 bits: SNR  49.1 dB   (theory ≈ 49.9)
  12 bits: SNR  73.1 dB   (theory ≈ 74.0)

+3 dB per octave of oversampling (8-bit quantizer, 24 kHz signal band):
   1x oversampled: in-band quantization noise  -53.1 dB   (+0.0 dB expected shift)
   4x oversampled: in-band quantization noise  -60.0 dB   (-6.0 dB expected shift)
  16x oversampled: in-band quantization noise  -70.6 dB   (-12.0 dB expected shift)


---
### 🕐 Session 2 of 2 — *Noise Shaping: the ΣΔ Loop* (~40 min)
**Goal:** put the quantizer in a feedback loop: same noise total, pushed out of band — 1 bit becomes hi-fi.
**Builds on:** Session 1.

---

## 3. The Loop That Cheats

💡 **Intuition.** Wrap the quantizer in feedback: integrate the error before quantizing, subtract the output. Solve the loop and the signal passes untouched while the quantization noise is multiplied by $(1 - z^{-1})$ — a **high-pass**: near DC the loop's memory cancels its own past mistakes. Total noise unchanged; its *location* moved to high frequencies you were going to [decimate away anyway](./Foundations_of_Signal_Processing_2.ipynb). First-order shaping buys 9 dB/octave; second-order, 15 — which is how a 1-bit stream at 64× oversampling delivers CD-quality audio (DSD, and virtually every audio ADC/DAC you own).

In [3]:
def sigma_delta_1bit(x, order=1):
    """1-bit ΣΔ modulator, first or second order."""
    v = np.zeros(len(x)); i1 = i2 = 0.0
    for k in range(len(x)):
        if order == 1:
            i1 += x[k] - v[k-1] if k else x[k]
            v[k] = 1.0 if i1 >= 0 else -1.0
        else:
            e = x[k] - v[k-1] if k else x[k]
            i1 += e
            i2 += i1 - v[k-1] if k else i1
            v[k] = 1.0 if i2 >= 0 else -1.0
    return v

M = 64
fs_os = fs_base*M
t_os = np.arange(2**18)/fs_os
x_os = 0.5*np.sin(2*np.pi*997*t_os)

plain_1bit = np.sign(x_os)                                # 1-bit quantizer, no loop
sd1 = sigma_delta_1bit(x_os, 1)
sd2 = sigma_delta_1bit(x_os, 2)

plt.figure(figsize=(9, 3))
for y, name in [(plain_1bit, "plain 1-bit"), (sd1, "ΣΔ 1st order"), (sd2, "ΣΔ 2nd order")]:
    f, P = sig.welch(y - x_os, fs=fs_os, nperseg=8192)
    plt.semilogx(f, 10*np.log10(P + 1e-16), label=name, linewidth=0.9)
plt.axvline(fs_base/2, color="k", linestyle=":", linewidth=1)
plt.text(fs_base/2, -60, " audio band edge", fontsize=7)
plt.legend(fontsize=8); plt.xlabel("Hz"); plt.ylabel("error PSD [dB/Hz]")
plt.title("noise SHAPING: same total error, swept out of the audio band")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2993110/1785049913.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [4]:
# redeem the promise: decimate each 1-bit stream back to 48 kHz and measure audio-band SNR
def audio_snr(y):
    dec = sig.decimate(sig.decimate(y, 8, ftype="fir"), 8, ftype="fir")   # 64x → 1x
    ref = sig.decimate(sig.decimate(x_os, 8, ftype="fir"), 8, ftype="fir")
    return 10*np.log10(np.var(ref)/np.var(dec - ref))

for y, name in [(plain_1bit, "plain 1-bit @64x"), (sd1, "ΣΔ 1st order"), (sd2, "ΣΔ 2nd order")]:
    snr = audio_snr(y)
    print(f"{name:18s}: audio-band SNR after decimation {snr:5.1f} dB  (~{(snr-1.76)/6.02:.1f} effective bits)")
print("\n→ ONE physical bit, second-order shaping, 64x oversampling ≈ a 12+ bit converter;")
print("  production designs add order, dither, and multibit stages to reach 16–24 bits")

plain 1-bit @64x  : audio-band SNR after decimation  -5.8 dB  (~-1.3 effective bits)
ΣΔ 1st order      : audio-band SNR after decimation  46.7 dB  (~7.5 effective bits)
ΣΔ 2nd order      : audio-band SNR after decimation  69.5 dB  (~11.3 effective bits)

→ ONE physical bit, second-order shaping, 64x oversampling ≈ a 12+ bit converter;
  production designs add order, dither, and multibit stages to reach 16–24 bits


## 4. Conclusion

Quantization noise has a fixed budget; oversampling dilutes it (+3 dB/octave, measured), and the ΣΔ loop *relocates* it (effective bits measured climbing with loop order). The dirty analog problem became a [multirate filtering](./Foundations_of_Signal_Processing_2.ipynb) problem — which is why converter datasheets read like DSP homework.

---
## Where next

- [Real-Time DSP](./Real_Time_DSP.ipynb) — where the decimated samples land.
- [Intro to FPGA](../Intro_FPGA/Intro_FPGA.ipynb) — CIC decimators: the hardware that does this at GHz.
- [Model Compression](../Intro_Mach_Learn/Model_Compression.ipynb) — the same quantization mathematics, aimed at neural weights.